# Olist E-Commerce Data Understanding

This notebook performs the initial data understanding of the Olist Brazilian E-Commerce Dataset.

The analysis includes:

- Dataset structure
- Row and column counts
- Column names
- Data types
- Missing values
- Duplicate records
- Unique values
- Initial identification of keys and relationships

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
RAW_DATA_PATH = "../data/raw/"

In [3]:
os.listdir(RAW_DATA_PATH)

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [4]:
customers = pd.read_csv(
    RAW_DATA_PATH + "olist_customers_dataset.csv"
)

geolocation = pd.read_csv(
    RAW_DATA_PATH + "olist_geolocation_dataset.csv"
)

order_items = pd.read_csv(
    RAW_DATA_PATH + "olist_order_items_dataset.csv"
)

order_payments = pd.read_csv(
    RAW_DATA_PATH + "olist_order_payments_dataset.csv"
)

order_reviews = pd.read_csv(
    RAW_DATA_PATH + "olist_order_reviews_dataset.csv"
)

orders = pd.read_csv(
    RAW_DATA_PATH + "olist_orders_dataset.csv"
)

products = pd.read_csv(
    RAW_DATA_PATH + "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    RAW_DATA_PATH + "olist_sellers_dataset.csv"
)

category_translation = pd.read_csv(
    RAW_DATA_PATH + "product_category_name_translation.csv"
)

In [5]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

In [6]:
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [7]:
dataset_summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
})

dataset_summary

,Dataset,Rows,Columns
0,customers,99441,5
1,geolocation,1000163,5
2,order_items,112650,7
3,order_payments,103886,5
4,order_reviews,99224,7
5,orders,99441,8
6,products,32951,9
7,sellers,3095,4
8,category_translation,71,2


## 2. Dataset Columns and Data Types

This section examines the columns and data types of each dataset to understand the structure and nature of the available attributes.

In [8]:
for name, df in datasets.items():
    print("=" * 80)
    print(f"{name.upper()}")
    print("=" * 80)
    print(df.dtypes)
    print()

CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

GEOLOCATION
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

ORDER_ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

ORDER_PAYMENTS
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

ORDER_REVIEWS
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
rev

In [9]:
structure_summary = []

for name, df in datasets.items():
    for column in df.columns:
        structure_summary.append({
            "Dataset": name,
            "Column": column,
            "Data Type": str(df[column].dtype),
            "Missing Values": df[column].isna().sum(),
            "Unique Values": df[column].nunique()
        })

structure_summary = pd.DataFrame(structure_summary)

structure_summary

,Dataset,Column,Data Type,Missing Values,Unique Values
0,customers,customer_id,str,0,99441
1,customers,customer_unique_id,str,0,96096
2,customers,customer_zip_code_prefix,int64,0,14994
3,customers,customer_city,str,0,4119
4,customers,customer_state,str,0,27
5,geolocation,geolocation_zip_code_prefix,int64,0,19015
6,geolocation,geolocation_lat,float64,0,717360
7,geolocation,geolocation_lng,float64,0,717613
8,geolocation,geolocation_city,str,0,8011
9,geolocation,geolocation_state,str,0,27


## 3. Missing Value Analysis

This section identifies missing values across all datasets.

Missing values are investigated before data preparation to determine whether they represent data quality issues or legitimate absence of information.

In [10]:
missing_summary = []

for name, df in datasets.items():
    total_rows = len(df)

    for column in df.columns:
        missing_count = df[column].isna().sum()

        if missing_count > 0:
            missing_summary.append({
                "Dataset": name,
                "Column": column,
                "Missing Values": missing_count,
                "Missing Percentage": round(
                    (missing_count / total_rows) * 100, 2
                )
            })

missing_summary = pd.DataFrame(missing_summary)

missing_summary.sort_values(
    by="Missing Percentage",
    ascending=False
)

,Dataset,Column,Missing Values,Missing Percentage
0,order_reviews,review_comment_title,87656,88.34
1,order_reviews,review_comment_message,58247,58.70
4,orders,order_delivered_customer_date,2965,2.98
6,products,product_name_lenght,610,1.85
5,products,product_category_name,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
3,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


## 4. Duplicate Record Analysis

This section checks whether complete duplicate rows exist in each dataset.

In [11]:
duplicate_summary = []

for name, df in datasets.items():
    duplicate_summary.append({
        "Dataset": name,
        "Duplicate Rows": df.duplicated().sum(),
        "Duplicate Percentage": round(
            (df.duplicated().sum() / len(df)) * 100, 2
        )
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

duplicate_summary

,Dataset,Duplicate Rows,Duplicate Percentage
0,customers,0,0.00
1,geolocation,261831,26.18
2,order_items,0,0.00
3,order_payments,0,0.00
4,order_reviews,0,0.00
5,orders,0,0.00
6,products,0,0.00
7,sellers,0,0.00
8,category_translation,0,0.00


## 5. Key and Relationship Analysis

This section identifies primary-key candidates, foreign-key relationships, and the cardinality between datasets.

Understanding these relationships is necessary for designing the analytical data warehouse and preventing incorrect joins and double-counting.

In [12]:
key_checks = {
    "customers.customer_id": customers["customer_id"],
    "customers.customer_unique_id": customers["customer_unique_id"],
    "orders.order_id": orders["order_id"],
    "products.product_id": products["product_id"],
    "sellers.seller_id": sellers["seller_id"],
    "reviews.review_id": order_reviews["review_id"],
}

for key, series in key_checks.items():
    print(
        f"{key:40} "
        f"Rows: {len(series):8} | "
        f"Unique: {series.nunique():8} | "
        f"Duplicates: {series.duplicated().sum():6}"
    )

customers.customer_id                    Rows:    99441 | Unique:    99441 | Duplicates:      0
customers.customer_unique_id             Rows:    99441 | Unique:    96096 | Duplicates:   3345
orders.order_id                          Rows:    99441 | Unique:    99441 | Duplicates:      0
products.product_id                      Rows:    32951 | Unique:    32951 | Duplicates:      0
sellers.seller_id                        Rows:     3095 | Unique:     3095 | Duplicates:      0
reviews.review_id                        Rows:    99224 | Unique:    98410 | Duplicates:    814


In [13]:
customer_frequency = (
    customers
    .groupby("customer_unique_id")
    .size()
    .value_counts()
    .sort_index()
)

customer_frequency

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [14]:
customers_per_unique = (
    customers
    .groupby("customer_unique_id")
    .size()
)

customers_per_unique.value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [15]:
print("Orders:", orders["order_id"].nunique())
print("Order items:", order_items["order_id"].nunique())
print("Payments:", order_payments["order_id"].nunique())
print("Reviews:", order_reviews["order_id"].nunique())

Orders: 99441
Order items: 98666
Payments: 99440
Reviews: 98673


In [16]:
order_item_counts = order_items.groupby("order_id").size()

print("Orders with 1 item:", (order_item_counts == 1).sum())
print("Orders with >1 item:", (order_item_counts > 1).sum())
print("Maximum items in an order:", order_item_counts.max())

Orders with 1 item: 88863
Orders with >1 item: 9803
Maximum items in an order: 21


In [17]:
review_id_counts = (
    order_reviews["review_id"]
    .value_counts()
)

review_id_counts[review_id_counts > 1]

review_id
c444278834184f72b1484dfe47de7f97    3
308316408775d1600dad81bd3184556d    3
2d6ac45f859465b5c185274a1c929637    3
3415c9f764e478409e8e0660ae816dd2    3
4219a80ab469e3fc9901437b73da3f75    3
                                   ..
8e954c79dc2fc426d5f17035e9bb22dd    2
b54d02f7f1520b3995f84b77ab0dacb3    2
870d856a4873d3a67252b0c51d79b950    2
9ea72755ef171e35b7b68ec0c8ed822c    2
d23bba9a2f1d16e5505a02e5968c1e68    2
Name: count, Length: 789, dtype: int64

In [18]:
duplicate_review_ids = review_id_counts[
    review_id_counts > 1
].index

order_reviews[
    order_reviews["review_id"].isin(duplicate_review_ids)
].sort_values("review_id")

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
40378,fe5c833752953fed3209646f1f63b53c,d3775e436e60258e62e678a0f68a0f8d,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07


In [19]:
review_key_check = (
    order_reviews
    .groupby(["review_id", "order_id"])
    .size()
)

print("Total rows:", len(order_reviews))

print(
    "Unique review_id + order_id combinations:",
    review_key_check.size
)

print(
    "Duplicate review_id + order_id combinations:",
    (review_key_check > 1).sum()
)

Total rows: 99224
Unique review_id + order_id combinations: 99224
Duplicate review_id + order_id combinations: 0


In [20]:
print("Orders:", orders["order_id"].nunique())
print("Order items:", order_items["order_id"].nunique())
print("Payments:", order_payments["order_id"].nunique())
print("Reviews:", order_reviews["order_id"].nunique())

Orders: 99441
Order items: 98666
Payments: 99440
Reviews: 98673


In [21]:
order_item_counts = order_items.groupby("order_id").size()

print("Orders with 1 item:", (order_item_counts == 1).sum())
print("Orders with >1 item:", (order_item_counts > 1).sum())
print("Maximum items in an order:", order_item_counts.max())

Orders with 1 item: 88863
Orders with >1 item: 9803
Maximum items in an order: 21


In [22]:
payment_counts = order_payments.groupby("order_id").size()

print("Orders with 1 payment:", (payment_counts == 1).sum())
print("Orders with >1 payment:", (payment_counts > 1).sum())
print("Maximum payments for an order:", payment_counts.max())

Orders with 1 payment: 96479
Orders with >1 payment: 2961
Maximum payments for an order: 29


In [23]:
payment_counts.value_counts().sort_index()

1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [24]:
review_counts = order_reviews.groupby("order_id").size()

print("Orders with 1 review:", (review_counts == 1).sum())
print("Orders with >1 review:", (review_counts > 1).sum())
print("Maximum reviews for an order:", review_counts.max())

Orders with 1 review: 98126
Orders with >1 review: 547
Maximum reviews for an order: 3


In [25]:
review_counts.value_counts().sort_index()

1    98126
2      543
3        4
Name: count, dtype: int64

In [26]:
missing_products = (
    ~order_items["product_id"].isin(products["product_id"])
).sum()

print("Order items referencing missing products:", missing_products)

Order items referencing missing products: 0


In [27]:
missing_sellers = (
    ~order_items["seller_id"].isin(sellers["seller_id"])
).sum()

print("Order items referencing missing sellers:", missing_sellers)

Order items referencing missing sellers: 0


In [28]:
missing_customers = (
    ~orders["customer_id"].isin(customers["customer_id"])
).sum()

print("Orders referencing missing customers:", missing_customers)

Orders referencing missing customers: 0


In [29]:
product_categories = set(
    products["product_category_name"]
    .dropna()
    .unique()
)

translated_categories = set(
    category_translation["product_category_name"]
    .unique()
)

untranslated_categories = product_categories - translated_categories

print("Untranslated categories:")
print(untranslated_categories)

Untranslated categories:
{'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


## 6. Data Grain Analysis

Data grain defines what one row represents in a dataset.

Understanding the grain of each dataset is essential for designing the analytical warehouse and avoiding double-counting during analysis.

In [30]:
grain_checks = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation
}

for name, df in grain_checks.items():
    print(f"{name}: {len(df):,} rows")

customers: 99,441 rows
orders: 99,441 rows
order_items: 112,650 rows
order_payments: 103,886 rows
order_reviews: 99,224 rows
products: 32,951 rows
sellers: 3,095 rows
geolocation: 1,000,163 rows


### Initial Grain Interpretation

- **Customers:** one row per customer record (`customer_id`)
- **Orders:** one row per order (`order_id`)
- **Order Items:** one row per product item within an order
- **Order Payments:** one row per payment record associated with an order
- **Order Reviews:** one row per review record associated with an order
- **Products:** one row per product (`product_id`)
- **Sellers:** one row per seller (`seller_id`)
- **Geolocation:** multiple geographic observations associated with a ZIP-code prefix
- **Category Translation:** one row per product-category translation

In [31]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_analysis = orders.copy()

for column in date_columns:
    orders_analysis[column] = pd.to_datetime(
        orders_analysis[column],
        errors="coerce"
    )

In [32]:
print(
    "Purchase date range:",
    orders_analysis["order_purchase_timestamp"].min(),
    "to",
    orders_analysis["order_purchase_timestamp"].max()
)

Purchase date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


In [33]:
orders_analysis["purchase_year"] = (
    orders_analysis["order_purchase_timestamp"].dt.year
)

orders_analysis["purchase_month"] = (
    orders_analysis["order_purchase_timestamp"].dt.month
)

orders_analysis["purchase_year"].value_counts().sort_index()

purchase_year
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

In [34]:
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [35]:
order_payments[[
    "payment_value",
    "payment_installments"
]].describe()

,payment_value,payment_installments
count,103886.000000,103886.000000
mean,154.100380,2.853349
std,217.494064,2.687051
min,0.000000,0.000000
25%,56.790000,1.000000
50%,100.000000,1.000000
75%,171.837500,4.000000
max,13664.080000,24.000000


In [36]:
order_reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [37]:
order_status_summary = (
    orders["order_status"]
    .value_counts()
    .rename_axis("Order Status")
    .reset_index(name="Orders")
)

order_status_summary["Percentage"] = (
    order_status_summary["Orders"]
    / len(orders)
    * 100
).round(2)

order_status_summary

,Order Status,Orders,Percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


## 7. Data Quality Findings

### Missing Values

- Review comment title has a high proportion of missing values.
- Review comment message also contains substantial missing values.
- Several order delivery timestamps contain missing values.
- Product category and product metadata contain 610 missing records.
- Product physical dimensions contain only 2 missing values each.

### Duplicate Records

- Most datasets contain no complete duplicate rows.
- The geolocation dataset contains a significant number of duplicate rows.
- Duplicate geolocation records require investigation based on the table's intended grain rather than automatic deletion.

### Key Findings

- `customer_id` is unique within the customer table.
- `customer_unique_id` is not unique and represents repeated customer identities across customer records.
- `order_id`, `product_id`, and `seller_id` behave as unique identifiers within their respective tables.
- `review_id` contains duplicate values and therefore requires careful treatment when modeling reviews.

### Relationship Findings

- Orders can contain multiple order items.
- Orders can contain multiple payment records.
- Orders can have review records.
- Order items reference products and sellers.
- Orders reference customers.

### Data Modeling Implications

The different grains of orders, order items, payments, and reviews mean these datasets should not be blindly joined together at row level, as this can cause double-counting.

## 8. Initial Business Questions

The analytical system should support questions such as:

### Sales Performance
- What are total sales over time?
- Which product categories generate the most revenue?
- Which products and sellers generate the highest sales?
- How does freight cost affect order value?

### Customer Analytics
- How many unique customers are there?
- What proportion of customers are repeat customers?
- Which customers generate the highest lifetime value?
- Which geographic regions have the highest customer activity?

### Order & Delivery
- What is the distribution of order statuses?
- What is the average delivery time?
- How often are orders delivered later than estimated?
- Which regions experience longer delivery times?

### Payment Analytics
- Which payment methods are most commonly used?
- What is the average payment value?
- How are installment payments distributed?

### Review Analytics
- What is the distribution of review scores?
- How do review scores vary by product category?
- Is there a relationship between delivery performance and review scores?